# NHANES III — standardized absolute risks, restricted mean survival and incremental discrimination (add-on)

**Run order.** Run `nhanes3_retina_ecg_mortality.ipynb` first. This notebook reads the analytic dataset that
notebook writes (`output/analytic_dataset.csv`) and does **not** re-download or re-derive anything.

**Why this add-on.** Hazard ratios computed in follow-up periods are conditional on surviving to the start of each
period and can change over time through depletion of susceptible persons alone (Hernán, *Epidemiology* 2010).
Absolute, covariate-standardized contrasts do not have this built-in selection problem and are easier to
interpret clinically.

**Estimands** (retinopathy vs no retinopathy, primary definition, Model 2 covariates, survey weights):

| Quantity | Definition |
|---|---|
| Standardized cumulative all-cause mortality at 5, 10, 15, 20, 25 y | g-formula: Cox model **stratified by retinopathy** (the exposure effect may vary freely over time) with common covariate effects; predictions under "all with" and "all without" retinopathy averaged over the weighted analytic sample |
| Standardized cumulative CVD mortality | cause-specific Cox models for CVD death and for death from other causes (competing event), both stratified by retinopathy; Aalen–Johansen-type cumulative incidence |
| Restricted mean survival time (RMST) difference, 10 and 25 y | months of life lost within the horizon |
| Restricted mean time lost (RMTL) to CVD death, 10 and 25 y | months lost to CVD death within the horizon |
| Incremental discrimination at 10 y | survey-weighted Harrell's C of Model 2 with vs without retinopathy (follow-up truncated at 10 y) |

**Uncertainty.** Design-based 95% CIs from the delete-one-PSU jackknife (98 replicates for NHANES III: every
replicate refits every model). 25 years is the shortest potential follow-up (examinations 1988–1994, follow-up
to 31 Dec 2019), so estimates up to 25 years are not affected by administrative censoring.

Runtime: about 3–8 minutes on a laptop. Outputs go to `output/absolute_risk/`.

In [19]:
# ------------------------------------------------------------------ 0. Setup
import sys, json, warnings, platform, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import nhanes3_tools as nt
import lifelines
for _f in ("survey_jackknife", "absolute_risk_suite", "harrell_c", "gformula_curves"):
    if not hasattr(nt, _f):
        raise ImportError("nhanes3_tools.py is older than this notebook: replace it with the v9 file delivered with it")

IN_FILE = HERE / "output" / "analytic_dataset.csv"
MAIN_OUT = HERE / "output"
OUT_DIR = MAIN_OUT / "absolute_risk"
FIG_DIR = OUT_DIR / "figures"
for _d in (OUT_DIR, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)
if not IN_FILE.exists():
    raise FileNotFoundError(f"{IN_FILE} not found - run nhanes3_retina_ecg_mortality.ipynb first (it writes this file)")

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160, "display.max_columns", 30, "display.max_rows", 200)
nt.set_figure_style()
RUNLOG = {"notebook": "nhanes3_absolute_risk.ipynb", "python": platform.python_version(), "pandas": pd.__version__,
          "numpy": np.__version__, "lifelines": lifelines.__version__}
print(RUNLOG)

{'notebook': 'nhanes3_absolute_risk.ipynb', 'python': '3.12.10', 'pandas': '2.3.3', 'numpy': '2.4.4', 'lifelines': '0.30.3'}


## CONFIG

In [20]:
HORIZONS = [5, 10, 15, 20, 25]   # years (25 y = shortest potential follow-up in NHANES III)
TAUS = [10, 25]                  # RMST / RMTL horizons, years
C_HORIZON = 10                   # incremental discrimination horizon, years
CURVE_STEP = 1 / 12              # years between plotted points of the standardized curves (monthly)
CURVE_MAX = 25                   # years
EXPOSURE = "ret_any"             # primary retinopathy definition, as set in the main notebook (RET_PRIMARY)

## 1. Load the analytic dataset and rebuild the Model 2 covariates exactly as in the main notebook

In [21]:
d = pd.read_csv(IN_FILE)
print(f"analytic dataset: {len(d):,} rows")

def prep(dd):
    # identical to prep() in the main notebook: explicit dummies, missing categories stay missing
    x = dd.copy()
    for lev in sorted(x["race_eth"].dropna().unique()):
        if lev == x["race_eth"].dropna().min():
            continue
        x[f"race_eth_{int(lev)}"] = np.where(x["race_eth"].isna(), np.nan, (x["race_eth"] == lev).astype(float))
    for lev in ["former", "current"]:
        x[f"smoke_{lev}"] = np.where(x["smoke"].isna(), np.nan, (x["smoke"] == lev).astype(float))
    return x

X = prep(d)
race_cols = [c for c in X.columns if c.startswith("race_eth_")]
M1 = ["age", "female"] + race_cols
M2 = M1 + ["edu_years", "bmi", "sbp", "bp_meds", "diabetes", "hba1c", "tc", "hdl", "egfr", "prior_mi",
           "smoke_former", "smoke_current"]
DESIGN_PSUS = d[["SDPSTRA6", "psu"]].drop_duplicates()          # same design object as the main notebook
cc = X.dropna(subset=["time_y", "death_all", "death_cvd", EXPOSURE, "w"] + M2).copy()
cc[EXPOSURE] = cc[EXPOSURE].astype(int)
print(f"Model 2 complete cases: n = {len(cc):,}; deaths = {int(cc['death_all'].sum()):,}; "
      f"CVD deaths = {int(cc['death_cvd'].sum()):,}; with retinopathy = {int(cc[EXPOSURE].sum()):,}")

# consistency check against the main notebook's Model 2 sample
main_cox = MAIN_OUT / "cox_retinopathy_mortality.csv"
if main_cox.exists():
    mc = pd.read_csv(main_cox)
    ref = mc[(mc["model"] == "M2") & (mc["outcome"] == "death_all")].iloc[0]
    same = (int(ref["n"]) == len(cc)) and (int(ref["events"]) == int(cc["death_all"].sum()))
    print(f"Main notebook Model 2 (all-cause): n = {int(ref['n']):,}, events = {int(ref['events']):,} -> "
          f"{'identical sample' if same else 'MISMATCH'}")
    if not same:
        raise RuntimeError("The Model 2 sample differs from the main notebook - rerun the main notebook, then this one")
RUNLOG.update({"n_model2": len(cc), "deaths": int(cc["death_all"].sum()), "cvd_deaths": int(cc["death_cvd"].sum()),
               "n_exposed": int(cc[EXPOSURE].sum())})

analytic dataset: 8,207 rows
Model 2 complete cases: n = 7,591; deaths = 4,676; CVD deaths = 1,756; with retinopathy = 550
Main notebook Model 2 (all-cause): n = 7,591, events = 4,676 -> identical sample


## 2. Point estimates

In [22]:
def suite(frame):
    return nt.absolute_risk_suite(frame, "time_y", "death_all", "death_cvd", EXPOSURE, M2, "w",
                                  horizons=HORIZONS, taus=TAUS, c_horizon=C_HORIZON,
                                  curve_step=CURVE_STEP, curve_max=CURVE_MAX)

t0 = time.time()
est_full = suite(cc)
print(f"one pass: {time.time() - t0:.1f} s; {len(est_full)} estimands")
print(est_full[[k for k in est_full.index if not k.startswith("curve_")]].round(2).to_string())

one pass: 3.4 s; 1854 estimands
all_F0_5            6.21
all_F1_5            8.85
all_RD_5            2.64
cause_CIF0_5        2.77
cause_CIF1_5        3.72
cause_RD_5          0.95
all_F0_10          15.20
all_F1_10          17.40
all_RD_10           2.20
cause_CIF0_10       6.26
cause_CIF1_10       7.59
cause_RD_10         1.32
all_F0_15          25.47
all_F1_15          28.11
all_RD_15           2.64
cause_CIF0_15      10.35
cause_CIF1_15      10.66
cause_RD_15         0.32
all_F0_20          35.99
all_F1_20          36.92
all_RD_20           0.93
cause_CIF0_20      13.63
cause_CIF1_20      15.21
cause_RD_20         1.58
all_F0_25          46.67
all_F1_25          45.53
all_RD_25          -1.15
cause_CIF0_25      16.33
cause_CIF1_25      17.03
cause_RD_25         0.71
all_RMST0_10      112.06
all_RMST1_10      109.57
all_dRMST_10       -2.49
cause_RMTL0_10      3.46
cause_RMTL1_10      4.68
cause_dRMTL_10      1.22
all_RMST0_25      236.85
all_RMST1_25      231.94
all_dRMST_25      

## 3. Design-based 95% CIs (delete-one-PSU jackknife)

In [23]:
t0 = time.time()
jk, R = nt.survey_jackknife(suite, cc, "w", "SDPSTRA6", "psu", design_psus=DESIGN_PSUS, progress_every=10)
RUNLOG.update({"jackknife_replicates": jk.attrs["n_replicates"], "jackknife_failed": jk.attrs["failed_replicates"],
               "jackknife_minutes": round((time.time() - t0) / 60, 1)})
if jk.attrs["failed_replicates"]:
    print("WARNING: some replicates failed; affected estimands have SE = NaN")
jk.to_csv(OUT_DIR / "jackknife_all_estimands.csv")

  jackknife replicate 10/98  (31 s elapsed, ~277 s left)
  jackknife replicate 20/98  (72 s elapsed, ~282 s left)
  jackknife replicate 30/98  (135 s elapsed, ~306 s left)
  jackknife replicate 40/98  (177 s elapsed, ~257 s left)
  jackknife replicate 50/98  (220 s elapsed, ~211 s left)
  jackknife replicate 60/98  (268 s elapsed, ~170 s left)
  jackknife replicate 70/98  (311 s elapsed, ~124 s left)
  jackknife replicate 80/98  (352 s elapsed, ~79 s left)
  jackknife replicate 90/98  (396 s elapsed, ~35 s left)
  jackknife replicate 98/98  (429 s elapsed, ~0 s left)
[jackknife] 98 replicates, 0 failed, 0 single-PSU strata skipped


In [24]:
def fmt(k, dec=1):
    r = jk.loc[k]
    return f"{r['estimate']:.{dec}f} ({r['lo95']:.{dec}f} to {r['hi95']:.{dec}f})"

rows = []
for h in HORIZONS:
    rows.append({"outcome": "all-cause death", "years": h,
                 "no_retinopathy_pct": fmt(f"all_F0_{h:g}"), "retinopathy_pct": fmt(f"all_F1_{h:g}"),
                 "risk_difference_pp": fmt(f"all_RD_{h:g}")})
for h in HORIZONS:
    rows.append({"outcome": "CVD death (competing risk)", "years": h,
                 "no_retinopathy_pct": fmt(f"cause_CIF0_{h:g}"), "retinopathy_pct": fmt(f"cause_CIF1_{h:g}"),
                 "risk_difference_pp": fmt(f"cause_RD_{h:g}")})
abs_tab = pd.DataFrame(rows)
abs_tab.to_csv(OUT_DIR / "absolute_risk_standardized.csv", index=False)
print("Standardized cumulative mortality, % (95% CI) and risk difference, percentage points (95% CI):")
print(abs_tab.to_string(index=False))

rm = []
for tau in TAUS:
    rm.append({"measure": "RMST, all-cause (months)", "horizon_years": tau,
               "no_retinopathy": fmt(f"all_RMST0_{tau:g}"), "retinopathy": fmt(f"all_RMST1_{tau:g}"),
               "difference_months": fmt(f"all_dRMST_{tau:g}")})
    rm.append({"measure": "RMTL to CVD death (months)", "horizon_years": tau,
               "no_retinopathy": fmt(f"cause_RMTL0_{tau:g}"), "retinopathy": fmt(f"cause_RMTL1_{tau:g}"),
               "difference_months": fmt(f"cause_dRMTL_{tau:g}")})
rm_tab = pd.DataFrame(rm)
rm_tab.to_csv(OUT_DIR / "restricted_mean_times.csv", index=False)
print("\nRestricted mean survival time / time lost to CVD death, months (95% CI):")
print(rm_tab.to_string(index=False))

cd = []
for lab, name in (("all", "all-cause death"), ("cause", "CVD death")):
    cd.append({"outcome": name, "horizon_years": C_HORIZON,
               "C_model2": fmt(f"C_{lab}_base", 3), "C_model2_plus_retinopathy": fmt(f"C_{lab}_plus", 3),
               "delta_C": fmt(f"dC_{lab}", 4)})
c_tab = pd.DataFrame(cd)
c_tab.to_csv(OUT_DIR / "incremental_discrimination.csv", index=False)
print("\nSurvey-weighted Harrell's C at 10 years (95% CI):")
print(c_tab.to_string(index=False))

Standardized cumulative mortality, % (95% CI) and risk difference, percentage points (95% CI):
                   outcome  years  no_retinopathy_pct     retinopathy_pct risk_difference_pp
           all-cause death      5    6.2 (5.5 to 7.0)   8.9 (6.6 to 11.1)   2.6 (0.6 to 4.7)
           all-cause death     10 15.2 (13.8 to 16.6) 17.4 (15.0 to 19.8)  2.2 (-0.0 to 4.4)
           all-cause death     15 25.5 (23.5 to 27.4) 28.1 (24.9 to 31.3)  2.6 (-0.4 to 5.7)
           all-cause death     20 36.0 (33.7 to 38.3) 36.9 (33.7 to 40.2)  0.9 (-2.0 to 3.9)
           all-cause death     25 46.7 (44.4 to 48.9) 45.5 (40.8 to 50.2) -1.1 (-5.3 to 3.0)
CVD death (competing risk)      5    2.8 (2.3 to 3.3)    3.7 (2.4 to 5.0)  1.0 (-0.3 to 2.2)
CVD death (competing risk)     10    6.3 (5.5 to 7.1)    7.6 (5.5 to 9.6)  1.3 (-0.8 to 3.5)
CVD death (competing risk)     15  10.3 (9.3 to 11.4)  10.7 (7.7 to 13.7)  0.3 (-2.6 to 3.2)
CVD death (competing risk)     20 13.6 (12.3 to 14.9) 15.2 (12.0 to 

In [25]:
# standardized curves with jackknife CIs (for figures and for the manuscript supplement)
pts = np.round(np.arange(0.0, CURVE_MAX + 1e-9, CURVE_STEP), 4)
curves = []
for p_ in pts:
    row = {"years": p_}
    for key in ("all_F0", "all_F1", "all_RD", "cause_CIF0", "cause_CIF1", "cause_RD"):
        r = jk.loc[f"curve_{key}_{p_:g}"]
        row.update({key: r["estimate"], f"{key}_lo95": r["lo95"], f"{key}_hi95": r["hi95"]})
    curves.append(row)
curves = pd.DataFrame(curves)
curves.to_csv(OUT_DIR / "standardized_curves.csv", index=False)
curves.iloc[::20].round(2)

,years,all_F0,all_F0_lo95,all_F0_hi95,all_F1,all_F1_lo95,all_F1_hi95,all_RD,all_RD_lo95,all_RD_hi95,cause_CIF0,cause_CIF0_lo95,cause_CIF0_hi95,cause_CIF1,cause_CIF1_lo95,cause_CIF1_hi95,cause_RD,cause_RD_lo95,cause_RD_hi95
0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
20,1.67,2.01,1.47,2.54,2.51,1.60,3.43,0.51,-0.54,1.56,1.00,0.62,1.39,1.55,0.95,2.16,0.55,-0.10,1.20
40,3.33,3.77,3.31,4.24,5.54,3.81,7.27,1.77,0.04,3.49,1.73,1.35,2.11,2.70,1.77,3.64,0.97,-0.01,1.95
60,5.00,6.21,5.45,6.96,8.85,6.57,11.13,2.64,0.58,4.71,2.77,2.29,3.25,3.72,2.40,5.05,0.95,-0.30,2.20
80,6.67,8.79,7.89,9.69,11.35,8.99,13.72,2.56,0.40,4.73,3.70,3.19,4.20,4.94,3.25,6.64,1.25,-0.44,2.93
100,8.33,11.68,10.61,12.76,15.06,12.61,17.50,3.37,1.08,5.66,4.94,4.31,5.57,6.44,4.48,8.39,1.49,-0.50,3.49
120,10.00,15.20,13.83,16.56,17.40,14.99,19.80,2.20,-0.04,4.44,6.26,5.46,7.07,7.59,5.53,9.64,1.32,-0.81,3.46
140,11.67,18.73,17.26,20.21,21.85,18.58,25.11,3.11,0.25,5.98,7.63,6.74,8.51,9.34,6.70,11.98,1.71,-0.71,4.14
160,13.33,22.12,20.38,23.86,24.16,20.66,27.66,2.03,-1.28,5.35,9.09,8.07,10.10,9.83,6.88,12.79,0.75,-2.10,3.59
180,15.00,25.47,23.50,27.44,28.11,24.94,31.28,2.64,-0.40,5.68,10.35,9.30,11.39,10.66,7.66,13.66,0.32,-2.59,3.22


## 4. Figure — standardized cumulative mortality and risk differences

Style: no titles; bold panel letters aligned with each y-axis title; statistics in the legend text of the
manuscript, not in the panels; every drawn reference line appears in the legend.

In [26]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
(axA, axB), (axC, axD) = axes
x = curves["years"].values
for ax, k0, k1, ylab in [(axA, "all_F0", "all_F1", "Cumulative mortality, %"),
                         (axB, "cause_CIF0", "cause_CIF1", "Cumulative CVD mortality, %")]:
    ax.step(x, curves[k0], where="post", color="#999999", lw=2.2, label="No retinopathy")
    ax.step(x, curves[k1], where="post", color="#000000", lw=2.2, label="Retinopathy")
    ax.set_xlim(0, CURVE_MAX); ax.set_ylim(bottom=0)
    ax.set_xlabel("Years since examination"); ax.set_ylabel(ylab)
    ax.set_xticks(range(0, CURVE_MAX + 1, 5))
for ax, k, ylab in [(axC, "all_RD", "Difference, percentage points"),
                    (axD, "cause_RD", "Difference, percentage points")]:
    ax.fill_between(x, curves[f"{k}_lo95"], curves[f"{k}_hi95"], step="post", color="#DDDDDD", lw=0,
                    label="95% CI")
    ax.step(x, curves[k], where="post", color="#000000", lw=2.2, label="Retinopathy minus no retinopathy")
    ax.axhline(0, color="#555555", lw=1, ls="--", label="No difference")
    for h in (10, 25):
        v = jk.loc[f"{'all' if k == 'all_RD' else 'cause'}_RD_{h:g}", "estimate"]
        ax.plot([h], [v], "o", color="#000000", ms=6, clip_on=False, zorder=5)
        at_edge = h >= CURVE_MAX
        ax.annotate(f"{v:.1f}", (h, v), textcoords="offset points", xytext=(-4 if at_edge else 0, -11),
                    ha="right" if at_edge else "center", va="top", fontsize=11,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9), zorder=6)
    ax.set_xlim(0, CURVE_MAX); ax.set_xticks(range(0, CURVE_MAX + 1, 5))
    ax.set_xlabel("Years since examination"); ax.set_ylabel(ylab)
h1, l1 = axA.get_legend_handles_labels()
h2, l2 = axC.get_legend_handles_labels()
fig.legend(h1 + h2, l1 + l2, loc="lower center", ncol=3, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.06, 1, 0.97), h_pad=2.5, w_pad=2.0)
nt.align_panel_letters(fig, [(axA, "A"), (axB, "B"), (axC, "C"), (axD, "D")])
for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"fig_standardized_risk.{ext}", dpi=300)
plt.close(fig)
print("saved", FIG_DIR / "fig_standardized_risk.png")

saved c:\claude\nhanes3_eye_heart\output\absolute_risk\figures\fig_standardized_risk.png


## 5. Run log

In [27]:
RUNLOG.update({"horizons": HORIZONS, "taus": TAUS, "c_horizon": C_HORIZON, "exposure": EXPOSURE,
               "covariates_model2": M2,
               "variance": "delete-one-PSU jackknife (JKn), centred at the full-sample estimate"})
with open(OUT_DIR / "run_log_absolute_risk.json", "w") as fh:
    json.dump(RUNLOG, fh, indent=2)
print(json.dumps(RUNLOG, indent=2))

{
  "notebook": "nhanes3_absolute_risk.ipynb",
  "python": "3.12.10",
  "pandas": "2.3.3",
  "numpy": "2.4.4",
  "lifelines": "0.30.3",
  "n_model2": 7591,
  "deaths": 4676,
  "cvd_deaths": 1756,
  "n_exposed": 550,
  "jackknife_replicates": 98,
  "jackknife_failed": 0,
  "jackknife_minutes": 7.2,
  "horizons": [
    5,
    10,
    15,
    20,
    25
  ],
  "taus": [
    10,
    25
  ],
  "c_horizon": 10,
  "exposure": "ret_any",
  "covariates_model2": [
    "age",
    "female",
    "race_eth_2",
    "race_eth_3",
    "race_eth_4",
    "edu_years",
    "bmi",
    "sbp",
    "bp_meds",
    "diabetes",
    "hba1c",
    "tc",
    "hdl",
    "egfr",
    "prior_mi",
    "smoke_former",
    "smoke_current"
  ],
  "variance": "delete-one-PSU jackknife (JKn), centred at the full-sample estimate"
}
